# Lab 03: Loops, Telemetry, and Evidence
Introduction to Programming — Space Engineering, 2026/2027  
Provisional version · Name: *write here* · Date: *write here*

In Lab 02 you made a decision for one spacecraft reading. Now you will apply rules to a sequence of readings and produce a useful summary.

By the end, aim to:
- combine `for` loops with `if/elif/else` and Boolean conditions;
- distinguish the current reading from information accumulated across readings;
- use `range()` without losing an endpoint;
- diagnose code that runs but gives the wrong result;
- write a precise specification and test an LLM's interpretation of it.

## Route

| Portion | Activities | Planning time |
|---|---|---:|
| Part A: in class | A1–A5: trace, modify, repair, build | 75 minutes |
| Class checkpoint | Paper exit exercise, clean run, save and commit | 15 minutes |
| Part B: at home | B1–B4: batch monitor, tests, prompt audit | 60–90 minutes |

Times are estimates. If you finish an activity early, use its stretch question rather than waiting. The optional extension is not required homework.

Try the short class traces and your first diagnosis yourself before using an LLM. At home, assistance is allowed: record what you asked, what you changed, and how you checked it. A detailed prompt helps communicate your intent; it does not guarantee correct code.

Use your existing course environment. This notebook needs only standard Python, with no downloads or additional packages. All telemetry here is synthetic and the rules are simplified teaching rules.

For each repair: identify a failing input → state expected and actual behavior → explain the cause → change the code → rerun the test. Repair the marked cells in place; the original faulty code remains in the supplied notebook and your Git history.

## Part A — in class
### A1. One loop, several decisions · about 12 minutes

A list holds an ordered sequence: `[20, 76, 90]`. A `for` loop assigns each value in turn to its loop variable and runs the indented block for that value. This is one *iteration*. The loop visits values in list order, including repeated values.

Here the outer indentation belongs to the loop; the deeper indentation belongs to a conditional inside that loop. The final `print()` is outside the loop.

Use these temperature rules in A1 and A2:
1. `OUTSIDE LIMITS` below −20°C or above 85°C.
2. Otherwise, `WARM` above 75°C.
3. Otherwise, `ACCEPTED`.

Trace just the first three readings by hand, then predict the final line. Run the cell once. Explain why 85°C and 86°C take different branches.

| Reading, °C | Predicted status | Observed status |
|---:|---|---|
| 20 | | |
| 85 | | |
| 86 | | |

Predicted final line:  
Why the boundary matters:

In [ ]:
temperatures_c = [20, 85, 86, -20, -21, 75, 76]

for temperature_c in temperatures_c:
    if temperature_c < -20 or temperature_c > 85:
        status = "OUTSIDE LIMITS"
    elif temperature_c > 75:
        status = "WARM"
    else:
        status = "ACCEPTED"
    print(temperature_c, "C:", status)

print("Assessment complete")

### Make a change that tests the structure

In the cell above:
1. Add a repeated value of 86 at the end of the list. Predict how many output lines it adds.
2. Move the final message inside the loop, run it, and explain the difference. Then restore it outside.
3. Restore the original list before continuing.

Discuss: if you moved `print(temperature_c, "C:", status)` outside the loop, what information would be lost?

Stretch: reorder the conditions so `temperature_c > 75` comes first. Which input exposes the problem? Restore the correct order afterward.

### A2. Remembering a count and a total · about 18 minutes

A counter starts before the loop and increases when an event occurs. `count = count + 1` reads the old value, adds one, and stores the result. A total uses the same pattern but adds a measurement instead of one.

Requirement: count every reading outside the inclusive interval −20°C to 85°C. Report one final count, including zero when no readings are outside limits.

The following code runs but is wrong. Before repairing it, trace `outside_count` after each iteration for `[90, 20, 91, 30]`. Find the line that prevents it remembering earlier readings.

In [ ]:
temperatures_c = [90, 20, 91, 30]
outside_count = 0

for temperature_c in temperatures_c:
    outside_count = 0
    if temperature_c < -20 or temperature_c > 85:
        outside_count = outside_count + 1

print("Readings outside limits:", outside_count)

My trace of the faulty count:  
Required final count:  
Actual faulty final count:  
Cause and repair:

After repairing, also test `[20, 30]`, `[90, 91]`, and `[]`. The last is an empty list: the loop body runs zero times. Record the results, then restore the original input.

| Input | Expected count | Observed count |
|---|---:|---:|
| `[90, 20, 91, 30]` | | |
| `[20, 30]` | | |
| `[90, 91]` | | |
| `[]` | | |

### Build a filtered summary

Now calculate the mean of readings inside the inclusive interval −20°C to 85°C. WARM readings are still inside this interval and must be included. Exclude every outside reading from both the count and total.

Plan where to put:
- initialization of a count and total;
- the condition deciding whether a reading contributes;
- the final division and output.

Use a loop, a count, and a total; do not use `sum()` for this exercise. If there are no included readings, print `No readings inside limits` instead of dividing by zero. `[]` is also a valid input.

For `[20, 86, -20, 80]`, the included count is 3, the total is 80°C, and the mean is approximately 26.67°C. A count is a number of readings, not a temperature.

In [ ]:
temperatures_c = [20, 86, -20, 80]

# Initialize your count and total.
# Loop over the values and include only readings inside limits.
# After the loop, print the count and either the mean or the no-readings message.

Check with `[86, -21]` and `[]` as well. Restore the original input afterward.

Why would dividing by the length of the original input list give the wrong mean?  
What happens if you move initialization inside the loop?

### A3. Repeating a calculation with `range()` · about 12 minutes

`range(start, stop)` supplies integers starting at `start` and stopping *before* `stop`. For example, `range(2, 5)` supplies 2, 3, 4. `range(4)` supplies 0, 1, 2, 3.

Requirement: begin at 50% charge. Simulate exactly five time steps, subtracting 7 percentage points at each step. Print steps 1 through 5 with the charge **after** that step's loss. These are fixed losses, not a 7% proportional decrease.

The code below runs, but it violates the requirement in two ways. Predict the final printed step and charge before fixing it. The required last line must describe step 5 at 15%.

In [ ]:
battery_charge = 50
loss_per_step = 7

for step in range(1, 5):
    print("Step", step, "charge:", battery_charge, "%")
    battery_charge = battery_charge - loss_per_step

Faulty last printed step and charge:  
Repair 1 and why it is needed:  
Repair 2 and why it is needed:

After fixing the simulation, add an `if/elif/else` chain inside the loop to print the battery status after each loss:
- CRITICAL below 15%;
- SAFE MODE below 25%;
- WARNING at most 35%;
- NOMINAL otherwise.

Use the newly calculated charge, not the previous step's charge. At 15%, which status is required?

Stretch: change the simulation to six steps. Explain why this is a simulation of evolving state, whereas A1 classified independent readings. Restore five steps afterward.

### A4. Plausible code, misleading result · about 13 minutes

Requirement: assess each battery reading independently using the same four battery rules from A3. Print exactly one status per reading. A previous reading must not affect the next status.

This code runs, and every comparison looks plausible. Trace the assignments to `status` for the reading 10 before running it. Then repair the code in place.

In [ ]:
battery_readings = [10, 30, 80]

for battery_charge in battery_readings:
    status = "NOMINAL"
    if battery_charge < 15:
        status = "CRITICAL"
    if battery_charge < 25:
        status = "SAFE MODE"
    if battery_charge <= 35:
        status = "WARNING"
    else:
        status = "NOMINAL"
    print(battery_charge, "%:", status)

Assignments to `status` for 10%, in order:  
Which `if` owns the `else` in the faulty code?  
My repair and its reason:

### A different mistake: carrying a status forward

A proposed replacement uses one decision chain, but initializes `status` only once and omits the final `else`. Predict what happens for `[10, 80]`, then run it.

Repair it so that every reading receives its own status. Test both `[10, 80]` and `[80, 10]`: reversing the inputs should reverse the corresponding statuses, without changing any reading's classification.

In [ ]:
battery_readings = [10, 80]
status = "NOMINAL"

for battery_charge in battery_readings:
    if battery_charge < 15:
        status = "CRITICAL"
    elif battery_charge < 25:
        status = "SAFE MODE"
    elif battery_charge <= 35:
        status = "WARNING"
    print(battery_charge, "%:", status)

Expected and faulty observed statuses for `[10, 80]`:  
Why a single reading of `[80]` would hide the mistake:  
My repair:

### A5. Build a battery report · about 20 minutes

Combine the ideas in a new cell. Use the four battery rules above. For each reading, print its value and one status. After the loop, print the number of readings in each category.

Requirements:
- Visit every reading, including repeated values.
- Each reading contributes to exactly one category.
- The four counts must add up to the number of input readings.
- An empty list produces four zero counts and no per-reading lines.
- Running the whole cell twice must produce the same output.

Before coding, complete the count predictions for the supplied data. Use ordinary variables for the four counters; dictionaries and functions are not needed.

| CRITICAL | SAFE MODE | WARNING | NOMINAL | Total readings |
|---:|---:|---:|---:|---:|
| | | | | |

My plan in 3–5 steps:
1.
2.
3.

In [ ]:
battery_readings = [80, 35, 25, 15, 14, 36, 24, 10, 80]

# Initialize four counters.
# For each reading: select one status, update its counter, and print.
# Print the four final counts outside the loop.

Test your report with `[10, 80]`, `[35, 35]`, and `[]`, then restore the supplied data. For each, predict the counts first.

| Input | Expected counts (critical, safe, warning, nominal) | Observed counts |
|---|---|---|
| `[10, 80]` | | |
| `[35, 35]` | | |
| `[]` | | |

Stretch: add a counter for all readings. Check that it agrees with the sum of the four category counts. Would this check alone detect a program that incorrectly calls every reading NOMINAL? Explain.

## Class checkpoint

### Save your starting point for the home work
- Repair all marked Part A cells and record the reasoning for each repair.
- Restart the kernel and run Part A in order. Unfinished Part B placeholders may run without errors; that does not mean they are complete.
- Save the notebook, inspect `git status`, and commit your class work separately with a message such as `Complete Lab 03 class work`.
- If class time runs short, record the agreed stopping point here: ______.

The home assignment below is separate. Do not silently add unfinished class activities to it; agree on any catch-up work in class.

## Part B — at home
### B1. From one reading to a sequence

Extend the Lab 02 health monitor. The rules are repeated here so that you can work from a complete specification.

For each independent reading, select the first applicable status:
1. CRITICAL if battery is below 15% **or** voltage is below 22 V.
2. SAFE MODE if battery is below 25%, voltage is below 24 V, temperature is outside the inclusive interval −20°C to 85°C, **or** communications are unavailable.
3. WARNING if battery is at most 35% **or** temperature is above 75°C.
4. NOMINAL otherwise.

All inputs in this task are valid numbers and Boolean values. You do not need to handle missing values, text in numeric fields, or invalid physical values. These records are observations; do not subtract a simulated battery loss from them.

### A small bridge: several values in one record

Each tuple below groups five fields in a fixed order:
`(label, battery_percent, voltage_v, temperature_c, comms_ok)`.

The list holds the records. The five names after `for` receive the five fields of the current record; this is called unpacking. This example is complete, so run it once to see the structure.

In [ ]:
example_records = [("Example", 80, 28, 20, True)]

for label, battery_charge, bus_voltage_v, panel_temperature_c, comms_ok in example_records:
    print(label, battery_charge, bus_voltage_v, panel_temperature_c, comms_ok)

### Plan before coding

Complete the expected status column from the written rules. Explain D, E, and F, where boundaries and priority interact.

| Label | Battery % | Voltage V | Temperature °C | Comms | Expected status |
|---|---:|---:|---:|---|---|
| A | 80 | 28 | 20 | True | |
| B | 10 | 28 | 20 | True | |
| C | 80 | 21 | 20 | True | |
| D | 15 | 24 | −20 | True | |
| E | 25 | 24 | 85 | True | |
| F | 35 | 28 | 76 | False | |
| G | 36 | 28 | 76 | True | |
| H | 80 | 28 | 20 | True | |

D:  
E:  
F:

Why is H useful even though it repeats the measurements in A?

### Implement one report

Use one loop to classify every record, print its label and status, and count each status. After the loop, print four category counts and the total number of records processed. Use the same counter pattern as A5.

A1–A5 used simpler rules for individual measurements. Here you must implement the complete four-input rules above. Reuse your Lab 02 decision chain if useful, then verify it against this specification.

For an empty list, print zero for every summary count. Repeated records count separately. Reset all counters each time the cell runs. Do not add functions, packages, or a user interface; concentrate on the loop and logic.

In [ ]:
records = [
    ("A", 80, 28, 20, True),
    ("B", 10, 28, 20, True),
    ("C", 80, 21, 20, True),
    ("D", 15, 24, -20, True),
    ("E", 25, 24, 85, True),
    ("F", 35, 28, 76, False),
    ("G", 36, 28, 76, True),
    ("H", 80, 28, 20, True),
]

# Initialize counters here.
for label, battery_charge, bus_voltage_v, panel_temperature_c, comms_ok in records:
    status = "NOT IMPLEMENTED"
    # Replace the placeholder with the full priority decision chain.
    # Update exactly one category count and the processed count.
    print(label, status)

# Print the final counts here.

### B2. Tests that can expose mistakes

Test the same implementation from B1 by temporarily replacing its `records` list. Do not write a separate classifier for each test. Record predictions before running and restore records A–H afterward.

Required tests:
1. The supplied A–H dataset: compare every status and all category counts with your predictions.
2. `[]`: no per-record output, all summary counts zero.
3. A critical reading followed by a nominal reading, then the reverse order.
4. Three readings with battery 14%, 15%, and 16%, keeping voltage 28 V, temperature 20°C, and communications available.
5. Three readings with voltage 21 V, 22 V, and 23 V, keeping battery 80%, temperature 20°C, and communications available.
6. One reading with both a critical battery and unavailable communications: priority must still be correct.
7. Your own small test that catches a different mistake. State the mistake it would catch.

These tests are evidence, not a proof of correctness. A count total can be correct even when individual statuses are wrong.

| Test | Exact inputs or labels | Expected statuses and counts | Observed result | Explanation of mismatch, if any |
|---|---|---|---|---|
| A–H | | | | |
| Empty input | | | | |
| Critical then nominal, and reverse | | | | |
| Battery boundary | | | | |
| Voltage boundary | | | | |
| Competing faults | | | | |
| My additional test | | | | |

What did my additional test detect that the other tests did not?

### B3. The context an LLM cannot guess

A colleague asks an LLM:

> Loop over the battery readings and count the alarms.

It proposes the code below. Treat this as a supplied hypothetical response; you do not need an LLM account.

```python
alarm_count = 0
for battery_charge in battery_readings:
    if battery_charge < 25:
        alarm_count = alarm_count + 1
print(alarm_count)
```

Without more context, “alarm” is ambiguous. Consider two possible requirements:

- Requirement A: count every reading below 25%.
- Requirement B: count the start of each run of consecutive readings below 25%. A low first reading starts a run; a reading of 25% or more ends it.

For `[30, 20, 18, 40, 22]`, predict the result under A and under B. Does the proposed code satisfy A, B, both, or neither? Explain why you need the intended requirement before declaring the code wrong.

Now write a replacement prompt for Requirement A. Include the input format and units, exact threshold, treatment of duplicates, empty input behavior, required output, and two examples with expected counts. Ask for a simple loop and an explanation. Use requirements you can verify, not just “make it correct.”

Then explain how you would change the prompt for Requirement B. Implementing B is optional.

Predicted count under A:  
Predicted count under B:  
Which requirement the proposed code satisfies, and why:

My replacement prompt for A:
> Write here.

What I would change for B:

Two tests I would use to check the response, and why:
1.
2.

### B4. Verification record and submission

If you used an LLM, record one useful interaction. If you did not, say so and use the supplied response in B3 for your audit. You do not need to invent an interaction.

| Question | Your record |
|---|---|
| What context did you provide, or what was missing from the supplied prompt? | |
| What suggestion did you accept, modify, or reject? | |
| Which exact input and expected output checked it? | |
| What uncertainty remained after that test? | |

In 2–4 sentences each:
1. Why do counters belong outside a loop's initialization while each reading needs its own status assignment?
2. Describe a case where the code ran successfully but violated the specification.
3. Why can a precise prompt improve an LLM answer without making testing unnecessary?

### Final checklist
- [ ] My B1 report follows the complete rules and reports every reading once.
- [ ] My tests include boundaries, an empty list, competing faults, and order changes.
- [ ] I recorded expected results independently of the implementation.
- [ ] I completed the prompt audit and verification record.
- [ ] I restored each main example's original input after testing.
- [ ] I restarted the kernel and ran all required cells in order, checking their outputs.
- [ ] I saved the notebook and committed the home work separately from the class checkpoint.

Use the course's existing submission procedure. Be ready to explain one repair and trace a short sequence without an LLM.

## Optional extension — consecutive events

Implement Requirement B from B3 using a Boolean variable such as `previous_was_low`. Decide what it should mean before the first reading. Update it on every iteration, including normal readings.

Test `[]`, `[20, 18]`, `[20, 30, 18]`, and `[25, 20, 25, 20]`. Predict each event count before running.

Explain why this task intentionally remembers the preceding reading, while the independent status classifier must not reuse the preceding status.

In [ ]:
# Optional: count starts of low-battery runs and test your implementation.